# Memory Management Patterns for Agentic AI (Claude)

This notebook demonstrates common memory patterns you can mix in production agents using Claude Haiku 4.5:
1. Thread-scoped short-term memory
2. Rolling summary compression
3. Structured profile memory
4. Episodic memory retrieval
5. Memory pruning and TTL-like cleanup

```mermaid
flowchart TD
    U[User Turn] --> STM[Thread Short-Term Memory]
    STM --> SUM[Rolling Summarizer]
    SUM --> LTM[(Long-Term Stores)]
    LTM --> P[Profile Memory]
    LTM --> E[Episodic Memory]
    E --> R[Retriever]
    R --> A[Agent Response]
    A --> TTL[Pruner TTL/Policy]
    TTL --> LTM
```

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Dict, List, TypedDict

from langchain.agents import create_agent
from langchain_anthropic import ChatAnthropic
from langgraph.checkpoint.memory import InMemorySaver

MODEL_NAME = "claude-haiku-4-5-20251001"
llm = ChatAnthropic(model=MODEL_NAME, temperature=0, max_tokens=1024)
checkpointer = InMemorySaver()

## Pattern 1: Thread-scoped short-term memory
This section shows how `create_agent` plus `InMemorySaver` keeps each `thread_id` isolated. Cell 2 is the setup; Cell 4 demonstrates that two threads retain different answers even though they ask the same follow-up question.

In [ ]:
agent = create_agent(model=llm, tools=[], checkpointer=checkpointer)

thread_a = {"configurable": {"thread_id": "thread-a"}}
thread_b = {"configurable": {"thread_id": "thread-b"}}

agent.invoke({"messages": [{"role": "user", "content": "My city is Chennai."}]}, config=thread_a)
agent.invoke({"messages": [{"role": "user", "content": "My city is Pune."}]}, config=thread_b)

r1 = agent.invoke({"messages": [{"role": "user", "content": "What is my city?"}]}, config=thread_a)
r2 = agent.invoke({"messages": [{"role": "user", "content": "What is my city?"}]}, config=thread_b)

print("Thread A ->", r1["messages"][-1].content)
print("Thread B ->", r2["messages"][-1].content)

## Pattern 2: Rolling summary compression
This section shows how older turns can be compressed into a compact summary before the agent context grows too large. Cell 6 defines the summarizer and then applies it to a sample conversation so you can see the compression output.

In [ ]:
def rolling_summary(llm_model, turns: List[str], keep_last: int = 2) -> str:
    if len(turns) <= keep_last:
        return ""
    old_turns = "\n".join(turns[:-keep_last])
    prompt = f"Summarize these conversation turns in 4 bullets with facts only:\n{old_turns}"
    return str(llm_model.invoke(prompt).content)

conversation = [
    "User: I am preparing for agent interviews."
    "Assistant: Focus on planning, tools, and memory patterns."
    "User: My strongest area is Python and systems design."
    "Assistant: Great, build one capstone pipeline project."
]

summary = rolling_summary(llm, conversation, keep_last=2)
print(summary)

## Pattern 3: Structured profile memory
This section stores stable user facts separately from the conversation log. Cell 8 defines the typed profile shape and writes a structured profile record that the agent could reuse across sessions.

In [ ]:
class UserProfile(TypedDict):
    name: str
    location: str
    goals: List[str]
    preferences: List[str]

profiles: Dict[str, UserProfile] = {}

profiles['u1'] = {
    'name': 'Anil',
    'location': 'India',
    'goals': ['Learn agentic AI', 'Ship local-first demos'],
    'preferences': ['Python', 'Claude', 'LangGraph'],
}

profiles['u1']

## Pattern 4: Episodic memory retrieval
This section stores time-stamped events and retrieves the most relevant ones for a query. Cell 10 defines the `Episode` record and the retrieval function, then shows the matching episodes for a Claude-focused question.

In [ ]:
@dataclass
class Episode:
    text: str
    tags: List[str]
    created_at: datetime

episodes: List[Episode] = [
    Episode("Built a planner-executor graph", ["planning", "langgraph"], datetime.now() - timedelta(days=4)),
    Episode("Added reflection critic loop", ["reflection", "quality"], datetime.now() - timedelta(days=2)),
    Episode("Integrated Claude model", ["claude", "inference"], datetime.now() - timedelta(days=1)),
]

def retrieve_episodes(query: str, max_items: int = 2) -> List[Episode]:
    q = query.lower()
    scored = []
    for ep in episodes:
        score = 0
        if q in ep.text.lower():
            score += 2
        score += sum(1 for t in ep.tags if t in q)
        scored.append((score, ep.created_at, ep))
    scored.sort(key=lambda x: (x[0], x[1]), reverse=True)
    return [x[2] for x in scored if x[0] > 0][:max_items]

hits = retrieve_episodes("How did we use Claude and planning?")
[(h.text, h.tags, h.created_at.isoformat(timespec="seconds")) for h in hits]

## Pattern 5: Memory pruning / TTL cleanup
This section removes stale entries so the memory store stays bounded and useful. Cell 12 implements TTL-style pruning and shows which episodes remain after the cutoff is applied.

In [ ]:
def prune_episodes(items: List[Episode], ttl_days: int = 3) -> List[Episode]:
    cutoff = datetime.now() - timedelta(days=ttl_days)
    return [ep for ep in items if ep.created_at >= cutoff]

pruned = prune_episodes(episodes, ttl_days=3)
[(e.text, e.created_at.isoformat(timespec="seconds")) for e in pruned]

## Suggested production strategy
1. Keep short-term memory in thread checkpoints.
2. Periodically summarize old turns into compact memory.
3. Store profile memory in a structured DB row or document.
4. Store episodic memory in a searchable index.
5. Apply TTL and quality-based pruning policies.